# 1. 단어 사전 기반 매핑 복원

## 1) Import

In [ ]:
import pandas as pd
from tqdm import tqdm

In [ ]:
!pip install triton
!pip install transformers==4.41.2
!pip install accelerate==0.31.0
!pip install -U bitsandbytes

## 2) Data Load

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/data"
data_path = os.path.join(base_path, 'miniproj/')

Mounted at /content/drive


In [ ]:
train = pd.read_csv(data_path+'train.csv', encoding = 'utf-8-sig')
test = pd.read_csv(data_path+'test.csv', encoding = 'utf-8-sig')

In [ ]:
#train = pd.read_csv('./train.csv', encoding = 'utf-8-sig')
#test = pd.read_csv('./test.csv', encoding = 'utf-8-sig')

## 3) 단어 사전 생성

In [ ]:
match_dict = {}

for input_text, output_text in zip(train['input'], train['output']):
    input_words = input_text.split()
    output_words = output_text.split()
    for iw, ow in zip(input_words, output_words):
        match_dict[iw] = ow

## 4) 변환 적용

In [ ]:
def replace_words(input_text, match_dict):
    words = input_text.split()
    replaced_words = [match_dict.get(word, word) for word in words]
    return " ".join(replaced_words)

In [ ]:
converted_reviews = test['input'].apply(lambda x: replace_words(x, match_dict)).tolist()

## 5) Submission

In [ ]:
submission = pd.read_csv(data_path+'sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
#submission = pd.read_csv('./sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
submission['output'] = converted_reviews

In [ ]:
submission.to_csv(data_path+'baseline_submission.csv', index = False, encoding = 'utf-8-sig')

In [ ]:
#submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')

# 2. LLM 활용 (Gemma)

## 1) Import

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

## 2) Data Load

In [ ]:
train = pd.read_csv(data_path+'train.csv', encoding = 'utf-8-sig')
test = pd.read_csv(data_path+'test.csv', encoding = 'utf-8-sig')

In [ ]:
#train = pd.read_csv('./train.csv', encoding = 'utf-8-sig')
#test = pd.read_csv('./test.csv', encoding = 'utf-8-sig')

In [ ]:
samples = []

for i in range(10):
    sample = f"input : {train['input'][i]} \n output : {train['output'][i]}"
    samples.append(sample)

## 3) Model Load

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type= 'nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model_id = 'beomi/gemma-ko-7b'
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config = bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

## 4) Inference

In [ ]:
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

restored_reviews = []


for index, row in tqdm(test.iterrows(), desc="진행도"):
    query = row['input']

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
                "Your task is to transform the given obfuscated Korean review into a clear, correct, "
                "and natural-sounding Korean review that reflects its original meaning. "
                "Below are examples of obfuscated Korean reviews and their restored forms:\n\n"
                f"Example, {samples}"
                "Spacing and word length in the output must be restored to the same as in the input. "
                "Do not provide any description. Print only in Korean."
            )
        },
        {
            "role": "user",
            "content": f"input : {query}, output : "
        },
    ]

    prompt = "\n".join([m["content"] for m in messages]).strip()


    outputs = pipe(
        prompt,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        max_new_tokens=len(query),
        eos_token_id=pipe.tokenizer.eos_token_id
    )

    generated_text = outputs[0]['generated_text']
    result = generated_text[len(prompt):].strip()


    restored_reviews.append(result)

진행도: 209it [1:38:50, 28.37s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.77 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.17 GiB is free. Process 8156 has 13.57 GiB memory in use. Of the allocated memory 10.84 GiB is allocated by PyTorch, and 2.60 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5) Submission

In [ ]:
submission = pd.read_csv('./sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
submission['output'] = restored_reviews

In [ ]:
submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')